# US 7 — Modeling specification

Run **Run All** in Cursor to complete Taiga tasks:

| Task | What this notebook does |
|------|-------------------------|
| **#60** | Load `final_merged_balanced_adjusted.csv` |
| **#61** | Confirm target + feature columns |
| **#62** | Verify `docs/modeling_spec.md` (metrics + split) |
| **#63** | Sign-off checklist (teammate review) |

Branch: `feature/modeling-baseline`

In [ ]:
from pathlib import Path

import pandas as pd


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "processed").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "data" / "processed").is_dir():
        return cwd.parent
    if cwd.name == "modeling" and (cwd.parent.parent / "data" / "processed").is_dir():
        return cwd.parent.parent
    return cwd


REPO_ROOT = _repo_root()
DATA_PATH = REPO_ROOT / "data" / "processed" / "final_merged_balanced_adjusted.csv"
SPEC_PATH = REPO_ROOT / "docs" / "modeling_spec.md"

TARGET = "electricity_demand_per_capita"
FEATURES = [
    "temperature_change_c",
    "co2_per_capita",
    "gdp",
    "population",
    "renewables_share_elec",
    "fossil_share_elec",
]

TRAIN_YEAR_MAX = 2018
TEST_YEAR_MIN = 2019
METRICS = ["RMSE", "MAE", "R²"]

print("Repo root:", REPO_ROOT)
print("Dataset:", DATA_PATH)
print("Spec:", SPEC_PATH)

## Task #60 — Load balanced dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing file:\n  {DATA_PATH}\nRun the master dataset pipeline first."
    )

df = pd.read_csv(DATA_PATH)
print("Task #60 — PASSED: dataset loaded")
print("Shape (rows, cols):", df.shape)
print("Year range:", int(df["year"].min()), "–", int(df["year"].max()))
print("Countries:", df["country"].nunique())
df.head()

## Task #61 — Confirm target and feature columns

In [ ]:
assert TARGET in df.columns, f"Missing target: {TARGET}"
missing = [c for c in FEATURES if c not in df.columns]
assert not missing, f"Missing features: {missing}"

model_cols = [TARGET] + FEATURES
coverage = df[model_cols].notna().mean().mul(100).round(1)
print("Task #61 — column coverage (% non-null):")
print(coverage.to_string())
assert coverage.eq(100).all(), "Some modeling columns have missing values"

print("\nTask #61 — PASSED")
print("Target:", TARGET)
print("Features:", FEATURES)

## Task #62 — Metrics and train/test split (documented in spec)

In [ ]:
assert SPEC_PATH.exists(), f"Create spec at: {SPEC_PATH}"
spec_text = SPEC_PATH.read_text(encoding="utf-8")

for token in ["RMSE", "MAE", "R²", "2001–2018", "2019–2022"]:
    assert token in spec_text, f"Spec missing: {token}"

train = df[df["year"] <= TRAIN_YEAR_MAX]
test = df[df["year"] >= TEST_YEAR_MIN]
print("Task #62 — PASSED: spec file contains metrics and split")
print("Metrics:", ", ".join(METRICS))
print(f"Train (years <= {TRAIN_YEAR_MAX}):", len(train), "rows")
print(f"Test (years >= {TEST_YEAR_MIN}):", len(test), "rows")
print("\nSpec path:", SPEC_PATH)

## Task #63 — Review with teammate

1. Share `docs/modeling_spec.md` and this notebook with your teammate.
2. After they agree, add both names to the **Sign-off** table in the spec.
3. In Taiga: close tasks #60–#63 and move **US 7** to **Done**.

In [ ]:
print("Task #63 — checklist")
print("[ ] Teammate read docs/modeling_spec.md")
print("[ ] Sign-off table updated in spec")
print("[ ] Commit + push feature/modeling-baseline")
print("[ ] US 7 marked Done in Taiga")